# 04 — Join imaging to program usages

Every per-compound imaging table keys on `public_compound_id`, the same
identifier as `core/usages/`. This notebook (1) joins the zel024 image
embeddings to the measured usages, (2) recomputes one row of
`annex_imaging/prediction_score_summary.csv` exactly from its fold-clean
prediction dump, and (3) fits a small in-sample ridge from image
embeddings to usages to illustrate the signal, clearly labeled in-sample.

Terms: an **embedding** is a fixed-length numeric vector summarizing an
image, produced by a pretrained image model. A **fold** is one of five
fixed compound groups; every shipped prediction is for a compound the
model never saw in training.

In [1]:
import sys
from pathlib import Path

import numpy as np
import pandas as pd

# Locate the package root (works whether the notebook runs from examples/
# or from the package root).
ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / "core" / "basis" / "basis_registry.json").exists())
sys.path.insert(0, str(ROOT / "src"))
print("package root located (all paths below are relative to it)")


package root located (all paths below are relative to it)


## Join compound embeddings to usages

In [2]:
CTX = "zel024_hek293"
emb = pd.read_parquet(ROOT / "annex_imaging" / "zel024_compound_embeddings.parquet")
compounds = pd.read_parquet(ROOT / "core" / "usages" / f"usages_{CTX}_compounds.parquet")
usages = np.load(ROOT / "core" / "usages" / f"usages_{CTX}.npy")

u = pd.DataFrame(usages, columns=[f"P{j + 1:02d}" for j in range(32)])
u.insert(0, "public_compound_id", compounds["public_compound_id"])
joined = emb.merge(u, on="public_compound_id", how="inner")
print(f"embeddings: {len(emb)} compounds | usages: {len(u)} | joined: {len(joined)}")
print("join coverage of imaged compounds:",
      f"{len(joined) / len(emb):.1%}")


embeddings: 10129 compounds | usages: 13914 | joined: 10114
join coverage of imaged compounds: 99.9%


## Reproduce one row of prediction_score_summary.csv exactly

`prediction_score_summary.csv` is derived from the fold-clean NPZ dumps:
each row's `mean_program_pearson` is the mean over the 32 programs of
the per-program Pearson between `y_pred_z` and `y_true_z` in that row's
file. Recomputing it from the NPZ should match to print precision.

In [3]:
scores = pd.read_csv(ROOT / "annex_imaging" / "prediction_score_summary.csv")
row = scores.query(
    "family == 'image_to_program' and context == 'zel024_hek293' and "
    "model == 'clip_mlp' and fold == 0 and seed == 0").iloc[0]

dump = np.load(ROOT / "annex_imaging" / "image_to_program_predictions"
               / "zel024_hek293" / "clip_mlp__fold0_seed0.npz")
y_true, y_pred = dump["y_true_z"], dump["y_pred_z"]
per_program = [np.corrcoef(y_pred[:, j], y_true[:, j])[0, 1] for j in range(32)]
recomputed = float(np.mean(per_program))
print(f"published mean_program_pearson: {row['mean_program_pearson']:.6f}")
print(f"recomputed from the NPZ dump: {recomputed:.6f}")
print(f"n_test_compounds: {row['n_test_compounds']} (dump rows: {len(y_true)})")


published mean_program_pearson: 0.137296
recomputed from the NPZ dump: 0.137296
n_test_compounds: 1983 (dump rows: 1983)


## In-sample illustration: image embeddings to usages

**This cell is an in-sample illustration, not a generalization number.**
A ridge model is fit and scored on the same joined compounds, with no
train/test split, so the value is optimistic by construction; it exists
to show that the signal is visible with twenty lines of numpy. The
fold-clean numbers are the ones in `prediction_score_summary.csv`
(CLIP + ridge reaches mean per-program r = 0.135 on held-out folds 1-4
in this context; every fold x seed cell is above its pairing null).

In [4]:
clip_cols = [c for c in joined.columns if c.startswith("clip_")]
X = joined[clip_cols].to_numpy(np.float64)
Y = joined[[f"P{j + 1:02d}" for j in range(32)]].to_numpy(np.float64)
X = (X - X.mean(0)) / X.std(0).clip(min=1e-8)
Y = (Y - Y.mean(0)) / Y.std(0).clip(min=1e-8)

ALPHA = 10.0
XtX = X.T @ X + ALPHA * np.eye(X.shape[1])
W = np.linalg.solve(XtX, X.T @ Y)
Y_hat = X @ W
r_in_sample = float(np.mean([np.corrcoef(Y_hat[:, j], Y[:, j])[0, 1] for j in range(32)]))
print(f"in-sample mean per-program Pearson (ridge on CLIP, no split): {r_in_sample:.3f}")
print("fold-clean reference (prediction_score_summary.csv, clip_ridge,"
      " folds 1-4): ~0.135 mean per-program Pearson")


in-sample mean per-program Pearson (ridge on CLIP, no split): 0.339
fold-clean reference (prediction_score_summary.csv, clip_ridge, folds 1-4): ~0.135 mean per-program Pearson


The fold-clean dumps remain the decision-grade evidence: they are
held-out predictions, and the imaging annex README's headline (image to
program is decision-grade where the cell line matches; predicting genes
directly reaches only 0.021-0.024) is computed from them. The
circularity rule for any new analysis: marker channels are prediction
targets, never input features (`docs/INTERPRETATION_LIMITS.md`).